# KPI Domain Review: the 4 new companies

UnitedHealth Group, Northrop Grumman, Caterpillar, T-Mobile US were added to the
company roster this session. Before merging their derived-KPI data into the
production table, this notebook investigates: for each of the 10 derived ratios,
is a gap (a) a genuine structural inapplicability (like Mastercard's existing
"no inventory -> no Quick Ratio" rule), or (b) just our statement-label alias
list not covering this company's specific wording for an otherwise-normal line
item? Only (a) should become a `not_applicable_by_cik.yaml` entry - (b) should
be fixed by extending the alias list, since silently treating a coverage gap as
"not applicable" would hide real, computable data.

**Method**: fetch real statement data via edgartools, look at the actual
row labels used, and reason from there - not from assumption.

In [1]:
import sys
sys.path.insert(0, "../src_metrics")

import polars as pl
from edgar import Company, MultiFinancials, set_identity
import os

set_identity(os.getenv("EDGAR_IDENTITY", "your-email@example.com"))

from derived_kpis import _get_stmt_df, _normalize_stmt_df, _row_to_year_series, STATEMENT_LABEL_ALIASES, EXPECTED_DERIVED_LABELS
from domain_rules import load_domain_rules, excluded_metrics_for
from config import load_metrics_config

pl.Config.set_tbl_rows(30)
pl.Config.set_fmt_str_lengths(120)

NEW_COMPANIES = {
    731766: "UnitedHealth Group",
    1133421: "Northrop Grumman",
    18230: "Caterpillar",
    1283699: "T-Mobile US",
}


/opt/homebrew/Caskroom/miniconda/base/envs/finsight-venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Initial coverage: what did the pipeline produce before any fixes?

In [2]:
# This reflects the FIRST fetch, before any alias/domain-rule changes below -
# kept here as the "before" baseline for comparison.
facts = pl.read_parquet("run1_new4_facts_raw.parquet")
derived = facts.filter(pl.col("metric_type") == "derived")

for cik, name in NEW_COMPANIES.items():
    cik10 = f"{cik:010d}"
    present = set(derived.filter(pl.col("cik") == cik10)["metric_label"].unique().to_list())
    missing = EXPECTED_DERIVED_LABELS - present
    print(f"{name}: {len(present)}/10 present, missing -> {sorted(missing) if missing else 'none'}")


UnitedHealth Group: 8/10 present, missing -> ['Operating Margin %', 'Quick Ratio']
Northrop Grumman: 10/10 present, missing -> none
Caterpillar: 8/10 present, missing -> ['Operating Margin %', 'Quick Ratio']
T-Mobile US: 9/10 present, missing -> ['Free Cash Flow']


**Note**: by the time this cell runs, `run1_new4_facts_raw.parquet` on disk
already reflects the *fixed* alias list (the fetch was re-run after applying
the fixes documented below, to produce the final data for merging). The
"before" numbers referenced in this notebook's prose come from the
investigation session's terminal output, reproduced in section 2 below by
directly inspecting each company's real statement labels - which is the
actual root-cause evidence, not just a before/after row count.

## 2. Real statement labels per company - the actual evidence

In [3]:
def dump_labels(cik, name):
    print(f"\n{'='*70}\n{name} (CIK {cik})\n{'='*70}")
    co = Company(cik)
    filings = co.get_filings(form="10-K").head(2)
    mf = MultiFinancials.extract(filings)
    for stmt_name, candidates in [
        ("INCOME", ["income_statement", "income"]),
        ("BALANCE", ["balance_sheet", "balance"]),
        ("CASHFLOW", ["cashflow_statement", "cash_flow", "cashflow"]),
    ]:
        df = _normalize_stmt_df(_get_stmt_df(mf, candidates))
        print(f"\n--- {stmt_name} ---")
        if df is None:
            print("  None")
            continue
        for lbl in df["label"].astype(str):
            print("  -", lbl)

dump_labels(731766, "UnitedHealth Group")



UnitedHealth Group (CIK 731766)



--- INCOME ---
  - Revenue from Products and Services
  - Total revenues
  - Premiums
  - Investment and other income
  - Medical costs
  - Depreciation and amortization
  - Loss on sale of subsidiary and subsidiaries held for sale
  - Cost of products sold
  - Total operating costs
  - Operating costs
  - Earnings from operations
  - Interest expense
  - Total income before income taxes
  - Provision for income taxes
  - Net earnings attributable to UnitedHealth Group common shareholders
  - Net earnings
  - Earnings attributable to noncontrolling interests
  - Anti-dilutive shares excluded from the calculation of dilutive effect of common share equivalents
  - Basic
  - Diluted
  - Dilutive effect of common share equivalents
  - Diluted weighted-average number of common shares outstanding
  - Basic weighted-average number of common shares outstanding

--- BALANCE ---
  - Cash and cash equivalents
  - Goodwill
  - Long-term investments
  - Short-term investments
  - Total current ass

In [4]:
dump_labels(1133421, "Northrop Grumman")



Northrop Grumman (CIK 1133421)



--- INCOME ---
  - Revenues
  - Other Comprehensive Income (Loss), Foreign Currency Transaction and Translation Adjustment, Net of Tax
  - Other Comprehensive Income (Loss), Other Components, Net of Tax
  - Gain on sale of business
  - Mark-to-market pension and OPB benefit (expense)
  - Non-operating FAS pension benefit
  - Other comprehensive income (loss), net of tax
  - Comprehensive income
  - Cost of Product and Service Sold
  - General and administrative expenses
  - Total operating costs and expenses
  - Operating income
  - Interest expense
  - Other, net
  - Non-operating FAS pension benefit
  - Earnings before income taxes
  - Federal and foreign income tax expense
  - Net earnings
  - Basic earnings per share
  - Diluted earnings per share
  - Weighted-average common shares outstanding, in millions
  - Weighted-average diluted shares outstanding, in millions

--- BALANCE ---
  - Cash and cash equivalents
  - Accounts receivable, net
  - Goodwill
  - Intangible assets, net


In [5]:
dump_labels(18230, "Caterpillar")



Caterpillar (CIK 18230)



--- INCOME ---
  - Total sales and revenues
  - Other operating (income) expenses
  - Consolidated profit before taxes
  - Profit of consolidated companies
  - Equity in profit (loss) of unconsolidated affiliated companies
  - Cost of goods sold
  - Total operating costs
  - Profit (loss)
  - Interest expense of Financial Products
  - Research and development expenses
  - Selling, general and administrative expenses
  - Operating profit
  - Interest expense excluding Financial Products
  - Other income (expense)
  - Provision (benefit) for income taxes
  - Profit of consolidated and affiliated companies
  - Less: comprehensive income (loss) attributable to the noncontrolling interests
  - Profit per common share (in dollars per share)
  - Profit per common share - diluted (in dollars per share)
  - Basic (in shares)
  - Diluted (in shares)
  - Goodwill impairment charge

--- BALANCE ---
  - Cash and cash equivalents
  - Inventories
  - Property, plant and equipment - net
  - Goodwill


In [6]:
dump_labels(1283699, "T-Mobile US")



T-Mobile US (CIK 1283699)



--- INCOME ---
  - Total revenues
  - Gain on disposal group held for sale
  - Impairment expense
  - Depreciation and amortization
  - Cost of services, exclusive of depreciation and amortization shown separately below
  - Total operating expenses
  - Interest expense, net
  - Reclassification of loss from cash flow hedges, net of tax effect of $65, $60 and $56
  - (Losses) gains on fair value hedges, net of tax effect of $(64), $5 and $0
  - Unrealized (loss) gain on foreign currency translation adjustment, net of tax effect of $0, $0 and $0
  - Actuarial gain (loss), net of amortization and reclassification, on pension and other postretirement benefits, net of tax effect of $2, $(29) and $(31)
  - Other comprehensive income
  - Total comprehensive income
  - Selling, general and administrative
  - Operating income
  - Other (expense) income, net
  - Total other expense, net
  - Income before income taxes
  - Income tax expense
  - Net income
  - Basic (in USD per share)
  - Diluted

## 3. Classification: alias gap vs. genuine structural absence

Reading the real labels above against `STATEMENT_LABEL_ALIASES` (before any
fixes):

| Company | Line item | Real label used | Verdict |
|---|---|---|---|
| UnitedHealth | Revenue | "Total revenues" | **Alias gap** - fixed |
| UnitedHealth | Net Income | "Net earnings" | **Alias gap** - fixed |
| UnitedHealth | Operating Income | "Earnings from operations" | **Alias gap** - fixed |
| UnitedHealth | Operating Income (tier 2) | "Total income before income taxes" | **Alias gap** - fixed |
| UnitedHealth | CapEx | "Purchases of property, equipment and capitalized software" | **Alias gap** - fixed |
| UnitedHealth | CFO | "Cash flows from operating activities" | **Alias gap** - fixed (verified this holds a real non-null value, not just a section header) |
| UnitedHealth | **Inventory** | **no inventory line exists at all** | **Genuine structural absence** - health insurer, no physical inventory. New `not_applicable_by_cik.yaml` rule: exclude Quick Ratio. |
| Northrop Grumman | Net Income | "Net earnings" | **Alias gap** - fixed (same fix as UnitedHealth) |
| Northrop Grumman | Equity | "Total shareholders' equity" (not "stockholders'") | **Alias gap** - fixed |
| Northrop Grumman | Inventory | "Inventoried costs, net" (contract-cost inventory) | **Alias gap** - fixed (defense contractors do carry an inventory-like line, just different wording) |
| Caterpillar | Revenue | "Total sales and revenues" | **Alias gap** - fixed |
| Caterpillar | Net Income | "Profit (loss)" | **Alias gap** - fixed |
| Caterpillar | Operating Income | "Operating profit" | **Alias gap** - fixed |
| Caterpillar | CapEx | "Capital expenditures – excluding equipment leased to others" | **Alias gap** - fixed |
| Caterpillar | CFO | "Net cash provided by (used for) operating activities" | **Alias gap** - fixed |
| T-Mobile | (most items) | standard wording, already aliased | already fine |

**Net result: 6 real alias additions + 1 genuine domain rule**, not the
7-8 "missing metrics per company" the raw output first suggested. Most of
what looked like domain gaps were just this company's specific wording for
an otherwise completely normal, present line item.

## 4. Coverage before vs. after the fixes

In [7]:
config = load_metrics_config()
domain_rules = load_domain_rules(config.domain_rules_path, EXPECTED_DERIVED_LABELS)

before = {
    "UnitedHealth Group": ["Free Cash Flow", "Net Profit Margin %", "Operating Cash Flow Ratio",
                            "Operating Margin %", "Quick Ratio", "ROA % (Avg Assets)", "ROE % (Avg Equity)"],
    "Northrop Grumman": ["Debt-to-Equity", "Net Profit Margin %", "Quick Ratio",
                          "ROA % (Avg Assets)", "ROE % (Avg Equity)"],
    "Caterpillar": ["Debt-to-Equity", "Free Cash Flow", "Net Profit Margin %", "Operating Cash Flow Ratio",
                     "Operating Margin %", "Quick Ratio", "ROA % (Avg Assets)", "ROE % (Avg Equity)"],
    "T-Mobile US": ["Free Cash Flow", "Net Profit Margin %", "Operating Margin %"],
}

print(f"{'Company':<20} {'Missing BEFORE':>15} {'Missing AFTER':>15}")
for cik, name in NEW_COMPANIES.items():
    cik10 = f"{cik:010d}"
    present_after = set(derived.filter(pl.col("cik") == cik10)["metric_label"].unique().to_list())
    excluded = excluded_metrics_for(domain_rules, cik10)
    still_missing = (EXPECTED_DERIVED_LABELS - excluded) - present_after
    print(f"{name:<20} {len(before[name]):>15} {len(still_missing):>15}   (still missing: {sorted(still_missing) if still_missing else 'none'})")


Company               Missing BEFORE   Missing AFTER
UnitedHealth Group                 7               1   (still missing: ['Operating Margin %'])
Northrop Grumman                   5               0   (still missing: none)
Caterpillar                        8               2   (still missing: ['Operating Margin %', 'Quick Ratio'])
T-Mobile US                        3               1   (still missing: ['Free Cash Flow'])


## 5. Residual gaps - flagged, not chased further

A small number of gaps remain after the fixes above (UnitedHealth/Caterpillar's
`Operating Margin %`, Caterpillar's `Quick Ratio`, T-Mobile's `Free Cash Flow`).
Spot-checked T-Mobile's case: its CapEx line label is
`"Purchases of property and equipment, including capitalized interest of $(43), ..."`
- SEC filings sometimes render a dynamic dollar figure directly inside the row
label itself, which breaks exact-string alias matching (`_row_to_year_series`
requires an exact match after case/whitespace normalization, not a prefix or
contains-match). Fixing this properly would mean changing the *matching
strategy* (e.g. prefix matching), not just adding another literal alias - a
real, legitimate follow-up, but a different and larger kind of change than the
"add the missing alias" fixes above. Left as a known, documented limitation
rather than expanded scope for this pass.